### 0. Colab setup

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/DuyTuong101/credit-risk-prediction.git"
REPO_DIR_NAME = "credit-risk-prediction"

def find_project_root(start):
    for candidate in [start, *start.parents]:
        if (
            (candidate / "modules" / "data_processing.py").is_file()
            and (candidate / "modules" / "model_training.py").is_file()
            and (candidate / "data").is_dir()
        ):
            return candidate
    return None

PROJECT_ROOT = find_project_root(Path.cwd())

if PROJECT_ROOT is None:
    clone_dir = Path("/content") / REPO_DIR_NAME
    if not clone_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(clone_dir)], check=True)
    PROJECT_ROOT = find_project_root(clone_dir)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot locate project root. Check the GitHub URL and repository structure."
    )

NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
os.chdir(NOTEBOOK_DIR)
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Notebook directory:", Path.cwd())


### 1. Import Libraries and Load Data

In [ ]:
import os
import sys
import urllib.request
import numpy as np
import pandas as pd
 
sys.path.append(os.path.abspath(".."))
from modules.data_processing import (
    clean_raw_data, split_data, build_preprocessing_pipeline,
    get_feature_names, save_features,
)
 
RAW_PATH = "../data/raw/credit_risk_dataset.csv"
RAW_URL = "https://raw.githubusercontent.com/DuyTuong101/credit-risk-prediction/main/data/raw/credit_risk_dataset.csv"     
FEATURES_DIR = "../features"
PROCESSED_DIR = "../data/processed"
for d in (os.path.dirname(RAW_PATH), FEATURES_DIR, PROCESSED_DIR):
    os.makedirs(d, exist_ok=True)
 
if not os.path.exists(RAW_PATH):                
    urllib.request.urlretrieve(RAW_URL, RAW_PATH)
 
df_raw = pd.read_csv(RAW_PATH)
print("Initial shape:", df_raw.shape)

### 2. Data Cleaning and Train/Test Split (Preventing Data Leakage)

In [ ]:
df_clean = clean_raw_data(df_raw)              
df_clean.to_csv(f"{PROCESSED_DIR}/credit_risk_cleaned.csv", index=False)
 

X_train, X_test, y_train, y_test = split_data(df_clean, target_col="loan_status")
 
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"Default rate Train: {y_train.mean():.4f} | Test: {y_test.mean():.4f}")


### 3. Run Baseline Configuration and Save Features

In [ ]:
num_cols = ['person_age', 'person_income', 'person_emp_length',
            'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']
ordinal_cols = ['loan_grade']
nominal_cols = ['person_home_ownership', 'loan_intent', 'cb_person_default_on_file']
 
configs = {
    "baseline":         dict(impute_strategy="median", scaler_type="standard"),
    "minmax_0_1":       dict(impute_strategy="median", scaler_type="minmax", feature_range=(0, 1)),
    "minmax_m1_1":      dict(impute_strategy="median", scaler_type="minmax", feature_range=(-1, 1)),
    "robust_logincome": dict(impute_strategy="median", scaler_type="robust", log_cols=["person_income"]),
    "knn_indicator":    dict(impute_strategy="knn", scaler_type="standard", add_missing_indicator=True),
    "standard_pca90":   dict(impute_strategy="median", scaler_type="standard", pca_variance=0.90),
    "standard_pca95":   dict(impute_strategy="median", scaler_type="standard", pca_variance=0.95),
    "minmax_pca95":     dict(impute_strategy="median", scaler_type="minmax", feature_range=(0, 1), pca_variance=0.95),
}
 
summary_rows = []
for name, cfg in configs.items():
    pipe = build_preprocessing_pipeline(num_cols, ordinal_cols, nominal_cols, **cfg)
    X_tr = pipe.fit_transform(X_train)           # fit CHỈ trên train
    X_te = pipe.transform(X_test)                # test chỉ transform
 
    assert not np.isnan(X_tr).any() and not np.isnan(X_te).any()
    assert X_tr.shape[1] == X_te.shape[1] == len(get_feature_names(pipe))
 
    save_features(name, pipe, X_tr, X_te, y_train, y_test, cfg,
                  out_dir=FEATURES_DIR, save_pipeline=(name == "baseline"))
 
    summary_rows.append({
        "Config": name,
        "Imputer": cfg.get("impute_strategy"),
        "Scaler": cfg.get("scaler_type"),
        "Log cols": ",".join(cfg.get("log_cols", [])) or "-",
        "Missing indicator": cfg.get("add_missing_indicator", False),
        "PCA variance": cfg.get("pca_variance") or "-",
        "n_features": X_tr.shape[1],
    })
 
print("Baseline features:", get_feature_names(
    build_preprocessing_pipeline(num_cols, ordinal_cols, nominal_cols).fit(X_train)))
display(pd.DataFrame(summary_rows))

### 4. Test Configurable Pipeline Options

In [ ]:
import glob, json
print(sorted(os.path.basename(p) for p in glob.glob(f"{FEATURES_DIR}/*")))
y_tr, y_te = np.load(f"{FEATURES_DIR}/y_train.npy"), np.load(f"{FEATURES_DIR}/y_test.npy")
for name in configs:
    a = np.load(f"{FEATURES_DIR}/X_train_{name}.npy"); b = np.load(f"{FEATURES_DIR}/X_test_{name}.npy")
    assert a.shape[0] == len(y_tr) and b.shape[0] == len(y_te) and a.shape[1] == b.shape[1]
print("OK: X/y align", len(configs), "configs")